# Etapa 7 · Grupos de alumnos con K-Means

Análisis que respalda `src/ml_grupos.py` (decisión **D57** en `docs/decisiones.md`).

**Pregunta:** ¿hay perfiles de alumnos que se repiten en las tres materias, según cómo usan el aula y cómo les va?
Sirve para que el tutor no mire 637 alumnos uno por uno, sino unos pocos grupos con una acción para cada uno.

**Variables** (una fila por alumno de la base y materia, a la fecha de corte 24/10/2025):
sesiones, minutos en la plataforma, semanas con actividad, avance (reporte de participación) y nota sobre 100.

El notebook solo **analiza**: todo el cálculo está en `src/ml_grupos.py`, que es lo que usa la app.
Para correrlo hace falta `pip install jupyter` (no está en `requirements.txt` porque la app no lo necesita).

In [1]:
import os
from pathlib import Path
if not Path("src").exists():          # el notebook está en notebooks/: subo a la raíz
    os.chdir("..")

import numpy as np
import pandas as pd
import plotly.express as px
import plotly.io as pio
from sklearn.metrics import adjusted_rand_score

from src import metricas as m
from src import ml_grupos as g
from src.config import GRUPOS_VARIABLES, GRUPOS_LOG

pio.renderers.default = "notebook_connected"
pd.set_option("display.width", 200)
pd.set_option("display.max_columns", 30)
d = m.cargar()
t = m.tabla_alumnos(d)
len(t), t["asignatura"].value_counts().to_dict()

(637,
 {'Álgebra': 217,
  'Probabilidad y Estadística': 216,
  'Algoritmos y Estructuras de Datos': 204})

## 1. Las variables

K-Means agrupa por distancia, así que importan dos cosas: la **escala** (sin estandarizar, los minutos —que llegan a 1.500— pesarían más que el avance, que va de 0 a 100) y la **asimetría** (unos pocos alumnos con muchísimas sesiones tiran los centroides).

In [2]:
t.groupby("asignatura")[GRUPOS_VARIABLES].median().round(1)

,sesiones,minutos,semanas_activas,avance,nota_pct
asignatura,,,,,
Algoritmos y Estructuras de Datos,22.0,157.1,9.0,66.0,34.0
Probabilidad y Estadística,32.0,241.0,10.0,66.0,47.4
Álgebra,26.0,144.5,10.0,80.0,75.6


In [3]:
asimetria = pd.DataFrame({
    "original": t[GRUPOS_VARIABLES].astype(float).skew(),
    "con log(1+x)": t[GRUPOS_VARIABLES].astype(float).apply(lambda s: np.log1p(s) if s.name in GRUPOS_LOG else s).skew(),
}).round(2)
asimetria

,original,con log(1+x)
sesiones,1.72,-1.05
minutos,1.94,-1.74
semanas_activas,-0.83,-0.83
avance,-0.65,-0.65
nota_pct,-0.15,-0.15


Sesiones y minutos están muy cargados a la derecha (asimetría > 1): les aplicamos `log(1 + x)`. El resto queda como está.
Ojo: el logaritmo no las deja simétricas, sino cargadas al otro lado (por los alumnos que casi no entraron, con 0 a 15 minutos). Lo que se busca es otra cosa: que la diferencia entre 1.000 y 1.500 minutos no pese más que la diferencia entre 10 y 200, que es la que importa para un tutor. En la sección 4 se ve que mejora la separación de los grupos.

Después **estandarizamos dentro de cada materia** (media 0, desvío 1). Así un alumno se compara con su curso: la mediana de minutos es 144 en Álgebra y 241 en PyE, y la de la nota es 76 en Álgebra y 34 en AED. Estandarizar con las tres materias juntas mezclaría "le va mal" con "cursa AED".

Las variables están bastante correlacionadas entre sí (Spearman):

In [4]:
t[GRUPOS_VARIABLES].astype(float).corr("spearman").round(2)

,sesiones,minutos,semanas_activas,avance,nota_pct
sesiones,1.00,0.84,0.85,0.65,0.56
minutos,0.84,1.00,0.70,0.57,0.46
semanas_activas,0.85,0.70,1.00,0.69,0.61
avance,0.65,0.57,0.69,1.00,0.76
nota_pct,0.56,0.46,0.61,0.76,1.00


Sesiones, minutos y semanas activas miden casi lo mismo (ρ ≈ 0,7–0,85): en la práctica, la actividad pesa 4 de 5 y la nota 1 de 5. Lo aceptamos porque es lo que pide el plan y porque el avance agrega información distinta (qué tanto del material recorrió, no cuánto tiempo estuvo). Una alternativa sería PCA antes del K-Means, pero haría los grupos más difíciles de explicar.

## 2. ¿Cuántos grupos? Silhouette

El **silhouette** mide, para cada alumno, qué tan cerca está de su grupo comparado con el grupo más cercano (de −1 a 1; más alto = grupos mejor separados). La **inercia** es la distancia total a los centroides: siempre baja al sumar grupos, por eso se mira dónde deja de bajar mucho (el "codo").

In [5]:
r = g.evaluar_k(d)
r

,k,silhouette,inercia,candidato,tamanos,elegido
0,2,0.475,1470.4,False,"[429, 208]",False
1,3,0.406,982.5,True,"[342, 211, 84]",True
2,4,0.332,820.4,True,"[296, 178, 115, 48]",False
3,5,0.299,716.2,True,"[250, 117, 116, 110, 44]",False
4,6,0.294,626.6,True,"[213, 128, 107, 93, 73, 23]",False


In [6]:
fig = px.line(r, x="k", y="silhouette", markers=True, title="Silhouette según la cantidad de grupos")
fig.add_vrect(x0=2.5, x1=6.5, fillcolor="lightgray", opacity=0.25, line_width=0, annotation_text="candidatos")
fig.show()
px.line(r, x="k", y="inercia", markers=True, title="Inercia (codo)").show()

- **k = 2** tiene el silhouette más alto, pero solo parte a los alumnos en "entran / no entran": eso ya lo dice el estado de actividad (D35) y no ayuda a decidir nada nuevo. Por eso se informa, pero no es candidato.
- Entre los candidatos (3 a 6), **k = 3** es el de mayor silhouette y además es donde está el codo de la inercia (de k = 2 a 3 baja un 33 %; de 3 a 4, un 16 %).
- Un silhouette de 0,4 indica una estructura **razonable pero no muy marcada**: los alumnos no forman grupos separados, sino un continuo de "más o menos enganchados". Los grupos son una forma útil de resumirlo, no tipos naturales.

### Estabilidad

Con otra semilla, ¿salen los mismos grupos? Y si muevo la fecha de corte, ¿se sigue eligiendo k = 3?

In [7]:
from src.config import GRUPOS_SEMILLA
from sklearn.cluster import KMeans
z = g.matriz(t)
base = KMeans(3, n_init=20, random_state=GRUPOS_SEMILLA).fit_predict(z)
{s: round(adjusted_rand_score(base, KMeans(3, n_init=20, random_state=s).fit_predict(z)), 3) for s in range(5)}

{0: 1.0, 1: 1.0, 2: 1.0, 3: 1.0, 4: 1.0}

In [8]:
filas = []
for f in ["2025-08-22", "2025-09-01", "2025-09-15", "2025-09-23", "2025-10-01", "2025-10-10", "2025-10-24"]:
    rr = g.evaluar_k(d, f)
    filas.append({"fecha_corte": f, "k_elegido": int(rr.loc[rr["elegido"], "k"].iloc[0]),
                  **{f"sil_k{k}": s for k, s in zip(rr["k"], rr["silhouette"])}})
pd.DataFrame(filas)

,fecha_corte,k_elegido,sil_k2,sil_k3,sil_k4,sil_k5,sil_k6
0,2025-08-22,3,0.458,0.387,0.381,0.310,0.308
1,2025-09-01,3,0.456,0.368,0.356,0.346,0.293
2,2025-09-15,3,0.445,0.358,0.357,0.332,0.271
3,2025-09-23,3,0.459,0.388,0.332,0.317,0.301
4,2025-10-01,3,0.474,0.413,0.335,0.308,0.292
5,2025-10-10,3,0.479,0.391,0.327,0.296,0.287
6,2025-10-24,3,0.475,0.406,0.332,0.299,0.294


El índice de Rand ajustado da 1 con cualquier semilla (mismos grupos) y k = 3 se elige en todas las fechas de corte desde que las tres materias tienen alguna evaluación. Antes de la primera evaluación de una materia (14/08 en AED, 17/08 en PyE, 21/08 en Álgebra) sus alumnos quedan **"Sin grupo"**: sin nota, el grupo solo reflejaría los primeros días de actividad.

## 3. Los grupos

Cada grupo se nombra según su **centroide**, medido en desvíos respecto del promedio de su materia: cuánta actividad tiene (promedio de las 4 variables de actividad) y qué nota.

In [9]:
c = g.centroides(d)
px.imshow(c.pivot(index="nombre_grupo", columns="descripcion", values="desvios").loc[["Comprometidos", "Intermitentes", "Desconectados"]],
          text_auto=True, color_continuous_scale="RdBu", zmin=-2, zmax=2, aspect="auto",
          title="Centroide de cada grupo (desvíos respecto del promedio de su materia)").show()

In [10]:
p = g.perfil_grupos(d)
p.drop(columns="descripcion_grupo")

,grupo,nombre_grupo,alumnos,pct,mediana_sesiones,mediana_minutos,mediana_semanas_activas,mediana_avance,mediana_nota_pct,nota_promedio,pct_activos,pct_libre,pct_promociona,pct_regular
0,1,Comprometidos,342,53.7,39.0,270.1,11.0,86.0,76.0,73.5,78.9,17.5,44.2,38.3
1,2,Intermitentes,211,33.1,18.0,134.7,7.0,47.0,20.0,25.3,27.5,87.2,6.6,6.2
2,3,Desconectados,84,13.2,4.0,15.4,2.0,9.0,0.0,7.8,4.8,98.8,0.0,1.2


In [11]:
for _, fila in p.iterrows():
    print(f"{fila['grupo']}. {fila['nombre_grupo']} ({fila['alumnos']} alumnos, {fila['pct']} %): {fila['descripcion_grupo']}")

1. Comprometidos (342 alumnos, 53.7 %): Entran seguido, avanzan con el material y les va bien. No necesitan intervención.
2. Intermitentes (211 alumnos, 33.1 %): Actividad y nota por debajo del promedio: entran de forma irregular y rinden poco. Todavía se pueden recuperar.
3. Desconectados (84 alumnos, 13.2 %): Casi no entran al aula y no rinden: riesgo alto de abandono, conviene contactarlos.


| Grupo | Alumnos | Perfil (medianas) | Qué hacer |
|---|---|---|---|
| **Comprometidos** | 342 (54 %) | 39 sesiones, 270 min, 11 de 12 semanas, avance 86 %, nota 76 | Nada: el 82 % ya está Regular o Promociona |
| **Intermitentes** | 211 (33 %) | 18 sesiones, 135 min, 7 semanas, avance 47 %, nota 20 | Es el grupo donde intervenir: siguen entrando (no abandonaron), pero el 87 % está Libre |
| **Desconectados** | 84 (13 %) | 4 sesiones, 15 min, 2 semanas, avance 9 %, nota 0 | Contactar: casi no entraron nunca. Es la población del predictor de abandono |

Los Intermitentes no se distinguen tanto por los minutos (−0,07 desvíos, casi el promedio) como por la **nota** (−0,68) y el **avance** (−0,48): están en el aula, pero no recorren el material ni rinden.

### Por materia y contra las otras clasificaciones

In [12]:
pm = g.perfil_grupos(d, por_materia=True)
pm.pivot(index="asignatura", columns="nombre_grupo", values="pct")[["Comprometidos", "Intermitentes", "Desconectados"]]

nombre_grupo,Comprometidos,Intermitentes,Desconectados
asignatura,,,
Algoritmos y Estructuras de Datos,47.1,40.2,12.7
Probabilidad y Estadística,54.2,32.4,13.4
Álgebra,59.4,27.2,13.4


In [13]:
ga = g.grupos_alumnos(d)
display(pd.crosstab(ga["nombre_grupo"], ga["condicion"], normalize="index").mul(100).round(1))
display(pd.crosstab(ga["nombre_grupo"], ga["estado_actividad"], normalize="index").mul(100).round(1))

condicion,Libre,Promociona,Regular
nombre_grupo,,,
Comprometidos,17.5,44.2,38.3
Desconectados,98.8,0.0,1.2
Intermitentes,87.2,6.6,6.2


estado_actividad,Activo,Inactivo,Sin actividad
nombre_grupo,,,
Comprometidos,78.9,21.1,0.0
Desconectados,4.8,86.9,8.3
Intermitentes,27.5,72.5,0.0


- La proporción de Desconectados es casi igual en las tres materias (13 %); lo que cambia es cuántos Intermitentes hay: AED tiene el 40 % (y la nota más baja), Álgebra el 27 %.
- Los grupos **no son lo mismo que la condición**: hay 60 Comprometidos Libres (entran mucho y les va mal en alguna evaluación) y 14 Intermitentes que promocionan.
- Tampoco son lo mismo que el estado de actividad: el 21 % de los Comprometidos está inactivo esta semana (venían bien y cortaron hace poco). Eso es justo lo que va a mirar el predictor de abandono.

In [14]:
px.scatter(ga, x="minutos", y="nota_pct", color="nombre_grupo", facet_col="asignatura", log_x=True,
           hover_data=["id_alumno", "semanas_activas", "avance"], opacity=0.7,
           category_orders={"nombre_grupo": ["Comprometidos", "Intermitentes", "Desconectados"]},
           title="Minutos (escala log) vs. nota, por grupo").show()

## 4. Alternativas que descartamos

Para comparar, el silhouette de otras formas de preparar los datos (k = 2 a 6):

In [15]:
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import silhouette_score
x = t[GRUPOS_VARIABLES].astype(float)
xl = x.apply(lambda s: np.log1p(s) if s.name in GRUPOS_LOG else s)
opciones = {
    "sin log, estandarizado con todas juntas": StandardScaler().fit_transform(x),
    "con log, estandarizado con todas juntas": StandardScaler().fit_transform(xl),
    "con log, estandarizado por materia (elegida)": g.matriz(t).values,
}
pd.DataFrame({nombre: {k: round(silhouette_score(Z, KMeans(k, n_init=20, random_state=42).fit_predict(Z)), 3)
                      for k in range(2, 7)} for nombre, Z in opciones.items()}).T

,2,3,4,5,6
"sin log, estandarizado con todas juntas",0.426,0.349,0.324,0.295,0.279
"con log, estandarizado con todas juntas",0.470,0.402,0.329,0.299,0.293
"con log, estandarizado por materia (elegida)",0.475,0.406,0.332,0.299,0.294


- El logaritmo mejora la separación (≈ +0,05 en k = 3).
- Estandarizar por materia da casi el mismo silhouette que con todas juntas, pero **se elige por interpretación**: un alumno queda "abajo" si está abajo de su curso.
- También probamos **un modelo por materia**: el silhouette es parecido (Álgebra algo más alto, AED más bajo), pero los grupos se llamarían distinto en cada materia y no se podrían comparar. Un solo modelo con las tres juntas da los mismos nombres en todo el tablero.

## 5. Limitaciones (para el informe)

- **Una sola cursada y ~210 alumnos por materia.** Los grupos describen esta cohorte; no hay forma de validarlos con otra.
- **Silhouette ≈ 0,4:** estructura moderada. Hay alumnos en el borde entre Comprometidos e Intermitentes que con pocos cambios pasarían de grupo.
- **El avance es una foto al 24/10 (D36):** con una fecha de corte anterior, el avance es el del final, no el de esa fecha. Afecta sobre todo a fechas tempranas.
- **La nota pesa en los grupos**, así que se van a parecer a la condición. Es a propósito (el plan pide agrupar por actividad y nota), pero un grupo no es un predictor: para anticipar el abandono está el otro modelo de la Etapa 7.
- **Los 0 por no rendir** (D15) empujan hacia abajo a quien no rindió nada; en los Desconectados casi todos tienen nota 0.